# 10 - Deep ensemble of independently-trained CRNNs (+ ReduceLROnPlateau)

TrOCR fine-tuning and a from-scratch CNN+Transformer-decoder hybrid were
both considered and dropped: with only 3,280 training lines, anything
that has to learn its own language model from scratch (or unlearn a
pretrained one) risks the failure mode already confirmed with TrOCR -
fluent but wrong output that ignores the image. Ensembling avoids that
risk entirely: it only combines CTC models we've already validated work
on this data.

**Ensemble method**: train 2 more copies of the winning recipe
(`CRNNResidual` + augmentation + elastic distortion, identical
hyperparameters, only the random seed changes) alongside the existing
`crnn_residual_aug_elastic.pt`, then decode by averaging **probabilities**
(not already-decoded text) across all 3 models before beam search + LM
fusion - keeps each model's uncertainty available to the LM step instead
of a hard majority vote on independently-decoded strings
(Lakshminarayanan, Pritzel & Blundell 2017, "Simple and Scalable
Predictive Uncertainty Estimation using Deep Ensembles", NeurIPS - see
RESOURCES.md). Deliberately not ensembling with the older, weaker
checkpoints (`crnn_baseline.pt` 0.568, `crnn_regularized.pt` 0.702) -
averaging a much weaker model in would likely dilute rather than help.

**Also added to these two runs: `ReduceLROnPlateau`**
(`torch.optim.lr_scheduler.ReduceLROnPlateau`, see RESOURCES.md) - halves
the LR only once the val score genuinely stops improving for 3 epochs,
instead of stopping training outright at `PATIENCE=8`. Different in kind
from `CosineAnnealingLR` (tried and rejected in notebooks/07, `-0.028`
vs. constant LR) - that schedule decayed preemptively on a fixed timeline
even while the model was still improving, which is exactly why it lost.
This one is reactive: it only acts once a plateau is real. Motivating
evidence from this project: the elastic-distortion continuation did hit a
genuine, clean plateau (early stopping at epoch 37, best epoch 29) -
a lower LR in that window might have squeezed out more before giving up.
`src/train.py::train_with_early_stopping` now takes an optional
`scheduler` argument (backward compatible - `None` by default, existing
behavior/tests unaffected).

This notebook reuses `src/features.py`/`src/model.py`/`src/train.py`
directly (now that the pipeline is graduated) instead of re-defining
everything again. **Two full ~50-epoch training runs below - run them
yourself, same as every other training run in this project.**

In [ ]:
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader

from src.config import MODELS_DIR, RANDOM_STATE, TARGET_COLUMN
from src.data import load_raw_data, split_train_val
from src.evaluate import combined_score
from src.features import AugmentedLineDataset, LineDataset, build_vocab, collate_lines
from src.model import CRNNResidual, CharNgramLM, beam_decode_batch, beam_search_decode
from src.train import BATCH_SIZE, DROPOUT, MAX_EPOCHS, PATIENCE, WEIGHT_DECAY, train_with_early_stopping

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Reload data, vocab, val sample - identical to every other notebook this project

In [ ]:
train_df = load_raw_data('train')
fit_df, val_df = split_train_val(train_df)
val_sample = val_df.sample(120, random_state=RANDOM_STATE).reset_index(drop=True)

char_to_idx, idx_to_char = build_vocab(fit_df[TARGET_COLUMN].tolist())
num_classes = len(char_to_idx) + 1
num_classes

In [ ]:
# Sanity check: every ensemble member must share the exact same
# char<->index mapping, or averaging their output probabilities is
# meaningless (index 7 could mean a different character per model).
_existing_ckpt = torch.load(MODELS_DIR / 'crnn_residual_aug_elastic.pt', map_location=device)
assert _existing_ckpt['char_to_idx'] == char_to_idx, 'vocab mismatch - cannot ensemble'
print('vocab matches the existing checkpoint - OK')

In [ ]:
val_ds = LineDataset(val_sample, char_to_idx)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_lines)
len(val_ds)

## Train ensemble member 2 (seed 43)

Identical recipe to `crnn_residual_aug_elastic.pt` - only `torch.manual_seed`
changes (affects weight init + augmentation randomness + DataLoader
shuffling order). ~50 epochs, same as every prior CRNN training run in
this project.

In [ ]:
SEED_2 = 43
torch.manual_seed(SEED_2)

fit_ds_2 = AugmentedLineDataset(fit_df, char_to_idx)
fit_dl_2 = DataLoader(fit_ds_2, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_lines)

model_2 = CRNNResidual(num_classes, dropout=DROPOUT).to(device)
optimizer_2 = torch.optim.AdamW(model_2.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
# Reactive LR decay: only lowers LR once the model genuinely plateaus for
# LR_PATIENCE epochs, unlike CosineAnnealingLR (tried and rejected in
# notebooks/07 - it decayed on a fixed schedule even while still improving).
scheduler_2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_2, mode='max', factor=0.5, patience=3
)

best_state_2, best_score_2, best_epoch_2, history_2 = train_with_early_stopping(
    model_2, optimizer_2, fit_dl_2, val_dl, idx_to_char, device,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, scheduler=scheduler_2,
)
model_2.load_state_dict(best_state_2)
print(f"seed {SEED_2}: best epoch {best_epoch_2}, greedy val score {best_score_2:.4f}")

torch.save(
    {'model_state_dict': model_2.state_dict(), 'char_to_idx': char_to_idx, 'idx_to_char': idx_to_char},
    MODELS_DIR / 'crnn_residual_aug_elastic_seed43.pt',
)

### Seed 43 was still improving at epoch 50 - continue instead of stopping

Score was still climbing (0.777 -> 0.778 epoch 49->50), LR never dropped
(the plateau scheduler never saw 3 flat epochs), and early stopping never
triggered - same situation the original CRNN hit on its first pass
(notebooks/05: 0.691 at epoch 50, still improving). That one was resolved
by continuing training from the checkpoint with a fresh optimizer instead
of a longer single run (momentum resets, weights don't) - same approach
here.

**Loads `model_2` from the saved checkpoint on disk rather than assuming
it's still in memory** - if the kernel restarted since the first seed-43
training cell ran, `model_2`/`fit_dl_2` etc. are gone, but the checkpoint
(50 epochs of real training) is already saved. Needs cells 1-5 (imports,
data, vocab, `val_dl`) to have been run in this kernel session first -
those are cheap/fast, no GPU training involved.

**`patience` raised to 15 (vs. the LR scheduler's `patience=3`)** - with
both patiences equal, early stopping could cut off the LR reduction's
benefit almost immediately (only ~5 epochs left for a second LR drop
before the 8-epoch early-stop counter fires, sharing the same "no
improvement" streak). 15 gives room for multiple LR reductions to actually
show their effect before giving up.

In [ ]:
SEED_2 = 43
CONTINUE_PATIENCE = 15  # >> the scheduler's patience=3, so LR reductions get a real chance

checkpoint_2 = torch.load(MODELS_DIR / 'crnn_residual_aug_elastic_seed43.pt', map_location=device)
assert checkpoint_2['char_to_idx'] == char_to_idx, 'vocab mismatch'
model_2 = CRNNResidual(num_classes, dropout=DROPOUT).to(device)
model_2.load_state_dict(checkpoint_2['model_state_dict'])

torch.manual_seed(SEED_2)
fit_ds_2 = AugmentedLineDataset(fit_df, char_to_idx)
fit_dl_2 = DataLoader(fit_ds_2, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_lines)

# Recompute live instead of trusting the printed 0.7778 from the earlier
# training cell's output - same practice as the original CRNN's
# continuation (notebooks/05), avoids a stale-baseline mismatch.
from src.train import _evaluate_greedy
best_score_2 = _evaluate_greedy(model_2, val_dl, idx_to_char, device)['score']
print(f"loaded checkpoint, current greedy val score: {best_score_2:.4f}")

optimizer_2b = torch.optim.AdamW(model_2.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
scheduler_2b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_2b, mode='max', factor=0.5, patience=3)

best_state_2, best_score_2, best_epoch_2, history_2b = train_with_early_stopping(
    model_2, optimizer_2b, fit_dl_2, val_dl, idx_to_char, device,
    max_epochs=MAX_EPOCHS, patience=CONTINUE_PATIENCE, initial_best_score=best_score_2, scheduler=scheduler_2b,
)
model_2.load_state_dict(best_state_2)
print(f"seed {SEED_2} continuation: best epoch {best_epoch_2} (0 = no improvement over the loaded "
      f"checkpoint), score {best_score_2:.4f}")

torch.save(
    {'model_state_dict': model_2.state_dict(), 'char_to_idx': char_to_idx, 'idx_to_char': idx_to_char},
    MODELS_DIR / 'crnn_residual_aug_elastic_seed43.pt',
)

## Train ensemble member 3 (seed 44) - same recipe, 100 epochs from the start

Seed 43 needed a continuation to stop still-improving at 50 epochs - start
seed 44 with a 100-epoch budget directly instead of repeating that in two
steps. Both ensemble members end up with comparable total training either
way.

In [ ]:
SEED_3 = 44
SEED44_MAX_EPOCHS = 100
SEED44_PATIENCE = 15  # >> the scheduler's patience=3, same reasoning as the seed-43 continuation
torch.manual_seed(SEED_3)

fit_ds_3 = AugmentedLineDataset(fit_df, char_to_idx)
fit_dl_3 = DataLoader(fit_ds_3, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_lines)

model_3 = CRNNResidual(num_classes, dropout=DROPOUT).to(device)
optimizer_3 = torch.optim.AdamW(model_3.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
scheduler_3 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_3, mode='max', factor=0.5, patience=3)

best_state_3, best_score_3, best_epoch_3, history_3 = train_with_early_stopping(
    model_3, optimizer_3, fit_dl_3, val_dl, idx_to_char, device,
    max_epochs=SEED44_MAX_EPOCHS, patience=SEED44_PATIENCE, scheduler=scheduler_3,
)
model_3.load_state_dict(best_state_3)
print(f"seed {SEED_3}: best epoch {best_epoch_3}, greedy val score {best_score_3:.4f}")

torch.save(
    {'model_state_dict': model_3.state_dict(), 'char_to_idx': char_to_idx, 'idx_to_char': idx_to_char},
    MODELS_DIR / 'crnn_residual_aug_elastic_seed44.pt',
)

## Individual scores - each member alone (beam + LM, same as the reference pipeline)

Diagnostic before ensembling: confirms all 3 members are independently
strong (not one weak outlier dragging the average down) and gives a
same-run baseline to compare the ensemble against, rather than the
possibly-stale recorded 0.826.

In [ ]:
def load_model(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    m = CRNNResidual(len(ckpt['char_to_idx']) + 1, dropout=DROPOUT).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m


models = [
    load_model(MODELS_DIR / 'crnn_residual_aug_elastic.pt'),
    load_model(MODELS_DIR / 'crnn_residual_aug_elastic_seed43.pt'),
    load_model(MODELS_DIR / 'crnn_residual_aug_elastic_seed44.pt'),
]

lm = CharNgramLM(fit_df[TARGET_COLUMN].tolist(), n=4, k=1e-3)

individual_scores = {}
for name, m in zip(['existing', f'seed{SEED_2}', f'seed{SEED_3}'], models):
    refs_i, hyps_i = [], []
    for imgs, _targets_cat, input_lengths, _target_lengths, _ids, texts in val_dl:
        imgs = imgs.to(device)
        log_probs = m(imgs).cpu()
        hyps_i.extend(beam_decode_batch(log_probs, input_lengths, idx_to_char, blank=0, beam_width=10, lm=lm, alpha=0.35))
        refs_i.extend(texts)
    individual_scores[name] = combined_score(refs_i, hyps_i)
    print(name, individual_scores[name])

## Ensemble decode: average probabilities across all 3 models, then beam+LM

In [ ]:
def ensemble_decode_batch(imgs, input_lengths, models, idx_to_char, lm=None, alpha=0.0, beam_width=10):
    """Averages softmax probabilities across independently-trained models
    (Lakshminarayanan, Pritzel & Blundell 2017 - see RESOURCES.md) before
    beam-search decoding, rather than voting on each model's own decoded
    text - keeps the ensemble's combined uncertainty available to the LM
    fusion step instead of collapsing it to hard per-model outputs first."""
    probs_sum = None
    with torch.no_grad():
        for m in models:
            log_probs = m(imgs)  # B x T x C, still on device
            probs = log_probs.exp()
            probs_sum = probs if probs_sum is None else probs_sum + probs
    avg_probs = (probs_sum / len(models)).clamp_min(1e-12).cpu()
    avg_log_probs = avg_probs.log()
    return [
        beam_search_decode(
            avg_log_probs[b], input_lengths[b], idx_to_char, blank=0, beam_width=beam_width, lm=lm, alpha=alpha
        )
        for b in range(avg_log_probs.size(0))
    ]


refs, hyps = [], []
for imgs, _targets_cat, input_lengths, _target_lengths, _ids, texts in val_dl:
    imgs = imgs.to(device)
    hyps.extend(ensemble_decode_batch(imgs, input_lengths, models, idx_to_char, lm=lm, alpha=0.35, beam_width=10))
    refs.extend(texts)

ensemble_score = combined_score(refs, hyps)
print('ensemble (3 models):', ensemble_score)
print('best individual member:', max(individual_scores.values(), key=lambda s: s['score']))

## Qualitative check - single existing model vs. ensemble, same lines

Same practice as every other real win this session (residual blocks,
augmentation, elastic distortion, beam+LM): read actual decoded lines
side by side before trusting the number. Confirms the gain is genuine
accuracy, not an artifact, and checks for any new failure mode the
aggregate score wouldn't show (e.g. the ensemble smoothing toward bland/
average predictions instead of getting specific words right).

In [ ]:
_existing_only = load_model(MODELS_DIR / 'crnn_residual_aug_elastic.pt')

_batch = next(iter(val_dl))
_imgs, _targets_cat, _input_lengths, _target_lengths, _ids, _texts = _batch
_imgs = _imgs.to(device)

with torch.no_grad():
    _single_log_probs = _existing_only(_imgs).cpu()
_single_hyps = beam_decode_batch(_single_log_probs, _input_lengths, idx_to_char, blank=0, beam_width=10, lm=lm, alpha=0.35)
_ensemble_hyps = ensemble_decode_batch(_imgs, _input_lengths, models, idx_to_char, lm=lm, alpha=0.35, beam_width=10)

for id_, ref, single_hyp, ens_hyp in zip(_ids, _texts, _single_hyps, _ensemble_hyps):
    marker = '' if single_hyp == ens_hyp else '  <-- differs'
    print(f"ID={id_}{marker}")
    print(f"  ref:      {ref!r}")
    print(f"  single:   {single_hyp!r}")
    print(f"  ensemble: {ens_hyp!r}")
    print()

## Gate check

Adopt only if the ensemble beats every individual member's same-run score
(not just the recorded 0.826) - otherwise the "improvement" could just be
that one member happened to train better this round, not a real ensembling
effect. If it wins: this needs its own inference path (loading 3
checkpoints instead of 1) - worth it only if the gain clears the same
noise floor established earlier this session (~0.001-0.002). If it
doesn't win, log it as a negative result next to the post-processing
findings; the reference pipeline stays the single `crnn_residual_aug_elastic.pt`
+ beam+LM at 0.826.